# Добавление флагов C40

Тематический этап подготовки Европы 2019–2023. Выполняйте ячейки сверху вниз в окружении проекта. Каждый ноутбук соответствует одному шагу; диагностика и применение согласованного изменения разделены. Входы, выходы и порядок описаны в [карте подготовки](README.md). Сохраненные результаты перенесены из выполненного общего ноутбука; сам перенос данных не преобразует.

## Шаг 8 – Согласованное добавление диагностических флагов C40

Добавляем в новую копию `europe/data/working/03_c40_diagnostic_flags.parquet` только три булевых поля. Все записи и существующие значения сохраняются. Правила основаны на полном разборе шага 7 и его [первичных источниках](../../reports/preparation/06_c40_review.md), а не на новых авиационных порогах.

| Поле | True означает |
| --- | --- |
| `c40_time_missing` | В C40_CROSS_TIME физический null. |
| `c40_after_first_attempt` | C40 строго позже первого элемента attempt_times. |
| `c40_after_landing` | C40 строго позже landing_time. |

Флаги не содержат null. Если C40 отсутствует, оба временных флага False: это не означает подтвержденную корректность C40. Сравнения выполняются в UTC, без допуска и искусственного сдвига времени. Первый элемент списка проверяется на равенство минимальному времени; пустые списки и null-элементы не допускаются для этого проверенного входа.

Это диагностические поля. Временные флаги используют сведения завершенного наблюдения и не являются входными признаками раннего прогноза. Источник и входная копия проверяются по SHA256 до и после. После записи сравниваем каждое прежнее значение, независимо пересчитываем флаги через NumPy и сверяем счетчики с шагом 7.

In [1]:
# Создаем согласованную копию с тремя диагностическими флагами C40
from pathlib import Path
import runpy
import json
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Скрипт сохраняет исходные поля и независимо проверяет флаги каждой строки
module = runpy.run_path(str(ROOT / 'europe/scripts/apply_c40_flags.py'))
c40_changes, c40_combinations = module['main']()
display(c40_changes)
print('Взаимоисключающие сочетания флагов:')
display(c40_combinations)
# Читаем журнал преобразования и подтверждаем сохранение строк и меток
c40_applied = json.loads((ROOT / 'europe/reports/preparation/07_c40_flags_applied.json').read_text(encoding='utf8'))
print('Новая рабочая копия:', c40_applied['output'])
print('Записей до и после:', c40_applied['rows_before'], c40_applied['rows_after'])
print('Отмеченных уходов до и после:', c40_applied['go_arounds_before'], c40_applied['go_arounds_after'])
print('Все существующие поля неизменны:', c40_applied['existing_columns_unchanged'])
print('Флаги независимо проверены для каждой строки:', c40_applied['independent_per_row_flag_validation'])

Добавлены флаги, строк: 1000000


Добавлены флаги, строк: 2000000


Добавлены флаги, строк: 3000000


Добавлены флаги, строк: 4000000


Добавлены флаги, строк: 5000000


...

Сохранены все записи и существующие поля: 7984254


,field,true_rows,false_rows,null_rows,go_arounds_with_true
0,c40_time_missing,1652382,6331872,0,7847
1,c40_after_first_attempt,87,7984167,0,45
2,c40_after_landing,42,7984212,0,0


Взаимоисключающие сочетания флагов:


,c40_time_missing,c40_after_first_attempt,c40_after_landing,rows,go_arounds
0,False,False,False,6331785,18361
1,True,False,False,1652382,7847
2,False,True,False,45,45
3,False,True,True,42,0


Новая рабочая копия: europe\data\working\03_c40_diagnostic_flags.parquet
Записей до и после: 7984254 7984254
Отмеченных уходов до и после: 26253 26253
Все существующие поля неизменны: True
Флаги независимо проверены для каждой строки: True


## Итог шага 8

Сохранены все 7 984 254 записи и 26 253 отмеченных ухода. Добавлены флаги: C40 отсутствует – 1 652 382 записи; после первой попытки – 87; после посадки – 42. Последние 42 входят в 87, а не образуют дополнительную группу. У 45 оставшихся записей с флагом после первой попытки отмечен уход; исправления этих случаев не выполнялись.

Неизменность всех прежних полей проверена полным сравнением. Исходный Parquet и предыдущая рабочая копия сохранены. Новые флаги не меняют тип самолета, время или GoA и не исключают аэропорты. [Журнал преобразования](../../reports/preparation/07_c40_flags_applied.json) хранит определения, счетчики и контрольные суммы.

Следующий разбор – повторяющиеся flight_id: отличаем повтор идентификатора от дубля наблюдения и возможного разделения одного полета на сегменты. Новое правило удаления или объединения заранее не задано.